In [4]:
import pandas as pd
import matplotlib.pyplot as plt

In [5]:
names = ['relience-industries','tcs','itc','infosys','airtel']

#---------below code is for the mistake i made----------------

# for name in names:
#     df = pd.read_csv(f'./{name}/fundamentals.csv')
#     print("before: ", df.columns)
#     df = df.drop(columns=['Unnamed: 0'], errors="ignore")
#     print("after : ", df.columns)

#     df.to_csv(f'./{name}/fundamentals.csv', index=False)

In [6]:
df = pd.read_csv(f'./{names[0]}/fundamentals.csv')

df.isnull().sum()

Year                   0
Sales +                0
Expenses +             0
Operating Profit       0
OPM %                  0
Other Income +         0
Interest               0
Depreciation           0
Profit before tax      0
Tax %                  1
Net Profit +           0
EPS in Rs              0
Dividend Payout %      1
Equity Capital         0
Reserves               0
Borrowings +           0
Other Liabilities +    0
Total Liabilities      0
Fixed Assets +         0
CWIP                   0
Investments            0
Other Assets +         0
Total Assets           0
dtype: int64

In [7]:
df[df.isnull().any(axis=1)]

,Year,Sales +,Expenses +,Operating Profit,OPM %,Other Income +,Interest,Depreciation,Profit before tax,Tax %,...,Equity Capital,Reserves,Borrowings +,Other Liabilities +,Total Liabilities,Fixed Assets +,CWIP,Investments,Other Assets +,Total Assets
12,TTM,"999,629","823,218","176,411",18%,"28,720","26,197","54,918","124,016",NaN,...,"13,532","863,748","374,593","787,073","2,038,946","1,096,375","217,097","256,335","469,139","2,038,946"


In [8]:
import pandas as pd
import re

names = ['relience-industries', 'tcs', 'itc', 'infosys', 'airtel']

# DB-ready column names
FUNDAMENTAL_COLS = [
    'year',
    'sales',
    'operating_profit',
    'net_profit',
    'eps_in_rs',
    'equity_capital',
    'reserves',
    'borrowings',
    'total_assets'
]

GROWTH_RATE = {
    'relience-industries': 7,
    'tcs': 10,
    'infosys': 10,
    'itc': 6,
    'airtel': 8
}

def normalize_column(col: str) -> str:
    col = col.lower()
    col = re.sub(r'[+%]', '', col)
    col = re.sub(r'\s+', '_', col)
    col = re.sub(r'_+', '_', col)
    return col.strip('_')

def clean_numeric(val):
    if pd.isna(val):
        return val
    return re.sub(r'[,+%]', '', str(val))

for name in names:
    df = pd.read_csv(f'./{name}/fundamentals.csv')

    # 1. Normalize headers (DB-ready)
    df.columns = [normalize_column(c) for c in df.columns]

    # 2. Drop TTM row
    df = df[df['year'] != 'TTM']

    # 3. Keep only required columns
    df = df[FUNDAMENTAL_COLS]

    # 4. Clean numeric columns
    for col in FUNDAMENTAL_COLS[1:]:
        df[col] = df[col].apply(clean_numeric)
        df[col] = pd.to_numeric(df[col], errors='coerce')

    # 5. Drop broken rows
    df.dropna(inplace=True)

    # 6. Save cleaned fundamentals (DB-ready)
    df.to_csv(f'./{name}/fundamentals.csv', index=False)

    # ---------- RATIOS ----------
    ratios = pd.DataFrame()
    ratios['year'] = df['year']

    equity = df['equity_capital'] + df['reserves']

    ratios['roe'] = df['net_profit'] / equity
    ratios['debt_equity'] = df['borrowings'] / equity
    ratios['opm'] = df['operating_profit'] / df['sales']

    # ---------- INTRINSIC VALUE ----------
    g = GROWTH_RATE[name]
    ratios['intrinsic_value'] = df['eps_in_rs'] * (8.5 + 2 * g)

    ratios.to_csv(f'./{name}/ratios.csv', index=False)

    print(f"\n{name.upper()} CLEANED FUNDAMENTALS")
    print(df.head(2))
    print("\nRATIOS")
    print(ratios)

print("Columns normalized, DB-ready, ratios generated.")



RELIENCE-INDUSTRIES CLEANED FUNDAMENTALS
       year   sales  operating_profit  net_profit  eps_in_rs  equity_capital  \
0  Mar 2014  433521             34935       22548       16.0            2940   
1  Mar 2015  374372             37449       23640       17.0            2943   

   reserves  borrowings  total_assets  
0    195747      138761        428843  
1    215556      168251        504486  

RATIOS
        year       roe  debt_equity       opm  intrinsic_value
0   Mar 2014  0.113485     0.698390  0.080584          360.000
1   Mar 2015  0.108193     0.770031  0.100032          382.500
2   Mar 2016  0.128958     0.840894  0.153278          495.000
3   Mar 2017  0.113128     0.824678  0.152349          484.875
4   Mar 2018  0.122928     0.817166  0.164563          607.500
5   Mar 2019  0.102908     0.794897  0.148240          652.500
6   Mar 2020  0.088787     0.790650  0.149605          652.500
7   Mar 2021  0.076751     0.398419  0.173255          877.500
8   Mar 2022  0.087038

In [8]:
import pandas as pd
names = ['relience-industries', 'tcs', 'itc', 'infosys', 'airtel']
for name in names:
    df = pd.read_csv(f'./{name}/fundamentals.csv')
    print('Before Modification',name)
    print(df)

    print('After Modification', name)
    df['year'] = pd.to_datetime(df['year'])

    print(df)

    df.to_csv(f'./{name}/fundamentals.csv')
    
    

Before Modification relience-industries
        year   sales  operating_profit  net_profit  eps_in_rs  equity_capital  \
0   Mar 2014  433521             34935       22548      16.00            2940   
1   Mar 2015  374372             37449       23640      17.00            2943   
2   Mar 2016  272583             41781       29861      22.00            2948   
3   Mar 2017  303954             46307       29833      21.55            2959   
4   Mar 2018  390823             64315       36080      27.00            5922   
5   Mar 2019  568337             84250       39837      29.00            5926   
6   Mar 2020  596679             89266       39880      29.00            6339   
7   Mar 2021  466307             80790       53739      39.00            6445   
8   Mar 2022  694673            108581       67845      44.87            6765   
9   Mar 2023  876396            142318       74088      49.00            6766   
10  Mar 2024  899041            162498       79020      51.45        

C:\Users\kanch\AppData\Local\Temp\ipykernel_2088\2422789346.py:9: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['year'] = pd.to_datetime(df['year'])
C:\Users\kanch\AppData\Local\Temp\ipykernel_2088\2422789346.py:9: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['year'] = pd.to_datetime(df['year'])
C:\Users\kanch\AppData\Local\Temp\ipykernel_2088\2422789346.py:9: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['year'] = pd.to_datetime(df['year'])
C:\Users\kanch\AppData\Local\Temp\ipykernel_2088\2422789346.py:9: UserWarning: Could not infer format, so each element will 

In [17]:
for name in names:
    df = pd.read_csv(f'./{name}/fundamentals.csv')

    print("Before modification: ")
    print(df.columns)

    df.drop(columns=['Unnamed: 0'], inplace=True)
    print("After modification: ")
    print(df.columns)

    df.to_csv(f'./{name}/fundamentals.csv', index=False)

Before modification: 
Index(['Unnamed: 0', 'year', 'sales', 'operating_profit', 'net_profit',
       'eps_in_rs', 'equity_capital', 'reserves', 'borrowings',
       'total_assets'],
      dtype='object')
After modification: 
Index(['year', 'sales', 'operating_profit', 'net_profit', 'eps_in_rs',
       'equity_capital', 'reserves', 'borrowings', 'total_assets'],
      dtype='object')
Before modification: 
Index(['Unnamed: 0', 'year', 'sales', 'operating_profit', 'net_profit',
       'eps_in_rs', 'equity_capital', 'reserves', 'borrowings',
       'total_assets'],
      dtype='object')
After modification: 
Index(['year', 'sales', 'operating_profit', 'net_profit', 'eps_in_rs',
       'equity_capital', 'reserves', 'borrowings', 'total_assets'],
      dtype='object')
Before modification: 
Index(['Unnamed: 0', 'year', 'sales', 'operating_profit', 'net_profit',
       'eps_in_rs', 'equity_capital', 'reserves', 'borrowings',
       'total_assets'],
      dtype='object')
After modification: 
Ind

In [3]:
import pandas as pd

names = ['relience-industries', 'tcs', 'itc', 'infosys', 'airtel']

for name in names:
    df = pd.read_csv(f'./{name}/ratios.csv')

    print(f'befor modification : {name} : \n')
    print(df)

    df['year'] = pd.to_datetime(df['year'])

    print('After thr modification:')
    print(df)

    df.to_csv(f'./{name}/ratios.csv', index = False)

befor modification : relience-industries : 

        year       roe  debt_equity       opm  intrinsic_value
0   Mar 2014  0.113485     0.698390  0.080584          360.000
1   Mar 2015  0.108193     0.770031  0.100032          382.500
2   Mar 2016  0.128958     0.840894  0.153278          495.000
3   Mar 2017  0.113128     0.824678  0.152349          484.875
4   Mar 2018  0.122928     0.817166  0.164563          607.500
5   Mar 2019  0.102908     0.794897  0.148240          652.500
6   Mar 2020  0.088787     0.790650  0.149605          652.500
7   Mar 2021  0.076751     0.398419  0.173255          877.500
8   Mar 2022  0.087038     0.409447  0.156305         1009.575
9   Mar 2023  0.103493     0.630928  0.162390         1102.500
10  Mar 2024  0.099587     0.442001  0.180746         1157.625
11  Mar 2025  0.096429     0.443920  0.171993         1158.075
After thr modification:
         year       roe  debt_equity       opm  intrinsic_value
0  2014-03-01  0.113485     0.698390  0.080584  

C:\Users\kanch\AppData\Local\Temp\ipykernel_8716\2162289303.py:11: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['year'] = pd.to_datetime(df['year'])
C:\Users\kanch\AppData\Local\Temp\ipykernel_8716\2162289303.py:11: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['year'] = pd.to_datetime(df['year'])
C:\Users\kanch\AppData\Local\Temp\ipykernel_8716\2162289303.py:11: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['year'] = pd.to_datetime(df['year'])
C:\Users\kanch\AppData\Local\Temp\ipykernel_8716\2162289303.py:11: UserWarning: Could not infer format, so each element w

In [3]:
import pandas as pd

names = ['relience-industries', 'tcs', 'itc', 'infosys', 'airtel']

needed_columns = [
    "date",
    "open_price",
    "high_price",
    "low_price",
    "close_price",
    "no_of_shares",
    "no__of_trades",
    "total_turnover_(rs_)"
]

for name in names:
    df = pd.read_csv(f'./{name}/prices.csv')

    df.columns = (
        df.columns
        .str.lower()
        .str.strip()
        .str.replace(" ", "_")
        .str.replace(".","_")
    )

    print(df.columns)

    df = df[needed_columns]

    df = df.rename(columns={'total_turnover_(rs_)': 'total_turnover', 'no__of_trades': 'no_of_trades'})

    print(df.columns)

    df['date'] = pd.to_datetime(df['date'])

    df.to_csv(f'./{name}/prices.csv', index= False)
    
    print(df.head())


Index(['date', 'open_price', 'high_price', 'low_price', 'close_price', 'wap',
       'no_of_shares', 'no__of_trades', 'total_turnover_(rs_)',
       'deliverable_quantity', '%_deli__qty_to_traded_qty', 'spread_high-low',
       'spread_close-open'],
      dtype='object')
Index(['date', 'open_price', 'high_price', 'low_price', 'close_price',
       'no_of_shares', 'no_of_trades', 'total_turnover'],
      dtype='object')
        date  open_price  high_price  low_price  close_price  no_of_shares  no_of_trades  total_turnover
0 2026-01-09     1466.95     1480.00    1465.00      1475.30        234309          9032    3.449429e+08
1 2026-01-08     1503.75     1503.75    1468.45      1470.30       2168603         43298    3.205026e+09
2 2026-01-07     1510.00     1519.95    1498.20      1504.10        710599         42333    1.071539e+09
3 2026-01-06     1575.55     1575.55    1497.05      1507.70        956028         36338    1.449102e+09
4 2026-01-05     1592.50     1611.20    1575.00     